In [ ]:
# Install Kaggle API
!pip install kaggle

# Create directory
!mkdir -p ~/.kaggle

# Copy kaggle.json (must be uploaded first)
!cp /content/kaggle.json ~/.kaggle/

# Change permissions
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API Successfully Configured!")

Kaggle API Successfully Configured!


In [ ]:
!git clone https://github.com/ultralytics/ultralytics /content/ultralytics
%cd /content/ultralytics

Cloning into '/content/ultralytics'...
remote: Enumerating objects: 75946, done.
remote: Counting objects: 100% (212/212), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 75946 (delta 172), reused 120 (delta 120), pack-reused 75734 (from 2)
Receiving objects: 100% (75946/75946), 40.65 MiB | 25.97 MiB/s, done.
Resolving deltas: 100% (57061/57061), done.
/content/ultralytics


In [ ]:
!pip install -e .
import torch
print("GPU:", torch.cuda.get_device_name(0))

Obtaining file:///content/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.234-0.editable-py3-none-any.whl size=23169 sha256=1d905425eabc3f561417bbc93843a8d8ce1e3c38d09704556837e69c766a9fb8
  Stored in directory: /tmp/pip-ephem-wheel-cache-a7rbx8bq/wheels/60/e0/59/e2f034f296abbdca5c21e3f5be76b9ca685f13c7bd17f8b58c
Successfully built ultralytics
GPU: Tesla T4


In [ ]:
import os, shutil

print("🧹 Cleaning datasets directory...")
if os.path.exists('/content/datasets'):
    shutil.rmtree('/content/datasets')

os.makedirs('/content/datasets')

%cd /content/datasets

🧹 Cleaning datasets directory...
/content/datasets


In [ ]:
print("⬇️ Downloading Brain Tumor Dataset...")
!kaggle datasets download -d pkdarabi/medical-image-dataset-brain-tumor-detection

⬇️ Downloading Brain Tumor Dataset...
Dataset URL: https://www.kaggle.com/datasets/pkdarabi/medical-image-dataset-brain-tumor-detection
License(s): Attribution 4.0 International (CC BY 4.0)
 97% 288M/297M [00:00<00:00, 728MB/s] 
100% 297M/297M [00:00<00:00, 786MB/s]


In [ ]:
print("📦 Unzipping...")
!unzip -q medical-image-dataset-brain-tumor-detection.zip
!rm medical-image-dataset-brain-tumor-detection.zip

📦 Unzipping...


In [ ]:
import os

print("🔍 Locating data.yaml...")

yaml_path = None

for root, dirs, files in os.walk('/content/datasets'):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        break

if yaml_path is None:
    raise FileNotFoundError("❌ data.yaml not found!")

dataset_root = os.path.abspath(os.path.dirname(yaml_path))

yaml_content = f"""
path: {dataset_root}
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['glioma', 'meningioma', 'pituitary']
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("✅ FIXED data.yaml at:", yaml_path)
print("📍 Dataset root:", dataset_root)

🔍 Locating data.yaml...
✅ FIXED data.yaml at: /content/datasets/BrainTumor/BrainTumorYolov11/data.yaml
📍 Dataset root: /content/datasets/BrainTumor/BrainTumorYolov11


In [ ]:
import re

tasks_path = "/content/ultralytics/ultralytics/nn/tasks.py"

bam_code = """
# --- BAM: Balanced Attention Mechanism START ---
# From: BAM: A Lightweight and Efficient Balanced Attention Mechanism for SISR
# Combines channel + spatial attention with balanced fusion

class BAM(nn.Module):
    def __init__(self, c1, c2=None, reduction=16):
        super().__init__()
        if c2 is None:
            c2 = c1

        mid = max(8, c1 // reduction)

        # ----- Channel Attention (CA) -----
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c1, mid, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, c1, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

        # ----- Spatial Attention (SA) -----
        self.sa = nn.Sequential(
            nn.Conv2d(c1, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

        # ----- Balanced Fusion weights -----
        self.alpha = nn.Parameter(torch.tensor(0.5))  # weight for CA
        self.beta  = nn.Parameter(torch.tensor(0.5))  # weight for SA

    def forward(self, x):
        ca_mask = self.ca(x)          # channel mask (B,C,1,1)
        sa_mask = self.sa(x)          # spatial mask (B,1,H,W)

        # balanced fusion: weighted sum of masks
        att = self.alpha * ca_mask + self.beta * sa_mask

        return x * (1 + att)          # residual attention
# --- BAM END ---
"""

print("🔧 Injecting BAM into tasks.py...")

# Read tasks.py
with open(tasks_path, "r") as f:
    lines = f.readlines()

# Skip if already present
if any("class BAM" in line for line in lines):
    print("ℹ️ BAM already exists — skipping.")
else:
    # Find BaseModel
    idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("class BaseModel"):
            idx = i
            break

    if idx is None:
        print("❌ ERROR: BaseModel not found.")
    else:
        # Insert above BaseModel
        new_lines = lines[:idx] + [bam_code + "\n"] + lines[idx:]
        with open(tasks_path, "w") as f:
            f.writelines(new_lines)
        print("✅ BAM inserted successfully.")

        # Register module inside YOLO
        with open(tasks_path, "r") as f:
            content = f.read()

        if "BAM" not in content:
            content = content.replace("if m in (", "if m in (BAM, ", 1)
            with open(tasks_path, "w") as f:
                f.write(content)
            print("✅ BAM registered.")
        else:
            print("ℹ️ BAM already registered.")

🔧 Injecting BAM into tasks.py...
✅ BAM inserted successfully.
ℹ️ BAM already registered.


In [ ]:
bam_yaml = """
# YOLOv8n + BAM Attention

nc: 3  # update depending on dataset

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

  # Insert BAM block here
  - [-1, 1, BAM, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]

  - [[16, 19, 22], 1, Detect, [nc]]
"""

with open("/content/ultralytics/yolov8n-bam.yaml", "w") as f:
    f.write(bam_yaml)

print("✅ Created yolov8n-bam.yaml")

✅ Created yolov8n-bam.yaml


In [ ]:
%cd /content/ultralytics
import os
os.environ["WANDB_DISABLED"] = "true"

!yolo train \
  data=/content/datasets/BrainTumor/BrainTumorYolov11/data.yaml \
  model=/content/ultralytics/yolov8n-bam.yaml \
  epochs=50 \
  imgsz=640 \
  batch=16 \
  name=YOLOv8_BAM

/content/ultralytics
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.234 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/BrainTumor/BrainTumorYolov11/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, h